In [1]:

from pathlib import Path
import pandas as pd

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd

ANON_PATH = PROJECT_ROOT / "data" / "anonymized" / "level1" / "adult_level1.csv"

if not ANON_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {ANON_PATH}. Update ANON_PATH to the actual file."
    )

df_level1 = pd.read_csv(
    ANON_PATH,
    skipinitialspace=True,
    dtype={"zip_code": "string"}
)

df_level1.columns = df_level1.columns.str.strip()

assert "income" in df_level1.columns
assert df_level1["income"].notna().all()

print("Dataset shape:", df_level1.shape)
print("Age examples:", df_level1["age"].head().tolist())
print("ZIP examples:", df_level1["zip_code"].head().tolist())
print("Dataset loaded successfully.")


Dataset shape: (32561, 16)
Age examples: ['30-39', '50-59', '30-39', '50-59', '20-29']
ZIP examples: ['938**', '245**', '132**', '460**', '420**']
Dataset loaded successfully.


In [2]:
# Prepare data and train Random Forest


from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Separate input features and target
X = df_level1.drop(columns=["income"])
y = df_level1["income"].astype(str).str.strip()

# Keep generalized and masked columns categorical
X["age"] = X["age"].astype("string")
X["zip_code"] = X["zip_code"].astype("string")

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

# Use the same split as the baseline
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Train Random Forest using baseline parameters
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_processed, y_train)
rf_predictions = rf_model.predict(X_test_processed)

# Calculate metrics
day11_result = {
    "experiment": "Level 1",
    "day": 11,
    "model": "Random Forest",
    "accuracy": accuracy_score(y_test, rf_predictions),
    "precision": precision_score(
        y_test, rf_predictions,
        pos_label=">50K", zero_division=0
    ),
    "recall": recall_score(
        y_test, rf_predictions,
        pos_label=">50K", zero_division=0
    ),
    "f1_score": f1_score(
        y_test, rf_predictions,
        pos_label=">50K", zero_division=0
    )
}

import pandas as pd
display(pd.DataFrame([day11_result]))

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Processed features:", X_train_processed.shape[1])
print("Random Forest training completed.")


,experiment,day,model,accuracy,precision,recall,f1_score
0,Level 1,11,Random Forest,0.861508,0.760972,0.61926,0.682841


Training rows: 26048
Test rows: 6513
Processed features: 1013
Random Forest training completed.


In [3]:
# Save the result


from pathlib import Path
import pandas as pd

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

output_path = RESULTS_DIR / "day11_level1_random_forest.csv"

pd.DataFrame([day11_result]).to_csv(output_path, index=False)

print("Results saved to:", output_path)
display(pd.read_csv(output_path))


Results saved to: d:\VS Code\Projects\python\Internship\results\day11_level1_random_forest.csv


,experiment,day,model,accuracy,precision,recall,f1_score
0,Level 1,11,Random Forest,0.861508,0.760972,0.61926,0.682841
